<a href="https://colab.research.google.com/github/yooongZa/AIFFEL_Quest_EPA/blob/main/LLM/%E1%84%85%E1%85%A2%E1%86%BC%E1%84%8E%E1%85%A6%E1%84%8B%E1%85%B5%E1%86%AB%E1%84%8B%E1%85%B3%E1%84%85%E1%85%A9_RAG_%E1%84%89%E1%85%B5%E1%84%8C%E1%85%A1%E1%86%A8%E1%84%92%E1%85%A1%E1%84%80%E1%85%B5_%5B%E1%84%91%E1%85%B3%E1%84%85%E1%85%A9%E1%84%8C%E1%85%A6%E1%86%A8%E1%84%90%E1%85%B3%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 랭체인으로 RAG 시작하기

해당 레슨은 Langchain으로 RAG를 구현하기 위해 필요한 각 컴포넌트인 Document Loaders, Text splitters, Text embeddings, Vectorstores, Retriever를 다룹니다

## Step 0 : 설치와 준비

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -U -q langchain langchain-openai
!pip install -U -q langchain-community langchain-core
!pip install -U langchain-text-splitters

In [ ]:
import os

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
#! curl ipinfo.io

In [ ]:
from langchain_openai import ChatOpenAI
import os
from google.colab import userdata

# OpenAI API를 사용하는 설정으로 변경
llm = ChatOpenAI(
    model="gpt-6-sol",
)

In [ ]:
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

## Step 1 : Document Loaders 사용해보기

In [ ]:
# 실습용 PDF(헤르만 헤세 《데미안》 영문판, 퍼블릭 도메인, 약 5.7MB) 를 내려받습니다. 파일이 이미 있고 비어 있지 않으면 건너뜁니다.
import os
if not os.path.exists("Demian.pdf") or os.path.getsize("Demian.pdf") == 0:
    !gdown 1oYqBihNheRgOH2JQ0zaXXLHuSuY27IPq -O Demian.pdf
!ls -lh Demian.pdf

-rw-r--r-- 1 root root 5.5M Sep 22 18:11 Demian.pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

In [ ]:
pages[0]

Document(metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': '/content/Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}, page_content='DEMIAN \n• \nDownloaded from https://www.holybooks.com')

In [ ]:
print(pages[10])

page_content='TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut F

In [ ]:
print(pages[10].page_content)

TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut Franz Kromer ga

```두 개의 세상
결국, 지나친 긴장감에 휩싸여 나는 말을 꺼내기 시작했다. 나는
내가 주인공으로 등장하는 긴 강도 이야기를 지어냈다. 어느 날 밤,
제분소 옆 구석에서 친구와
나는 자루 한 가득 사과를 훔쳤는데, 그건
평범한 사과가 아니라 피핀 사과, 그것도 최고급
황금빛 피핀 사과였다. 나는 그 순간의 위험에서 벗어나기 위해
그 이야기에 의지하고 있었고, 이야기를 지어내고
전하는 데 아무런 어려움도 느끼지 못했다. 말이 너무 빨리 끊겨서
어쩌면 더 나쁜 상황에 휘말리지 않기 위해, 나는
이야기하는 능력을 마음껏 발휘했다. 나는 우리 중 한 명이
항상 경계를 서고, 다른 한 명은 나무 위에 앉아
사과를 아래로 던졌다고 말했고, 자루가 너무 무거워져서
결국 자루를 열어 절반은 남겨두고 와야 했지만,
30분 후에 다시 돌아와 그 사과들도
다 가져왔다고 했다.
이야기가 끝날 때쯤 박수라도 받을 수 있기를 바랐는데; 나는
결국 내 이야기의 열기에 휩쓸려
말솜씨에 취해 열을 올리고 있었다. 두 어린 소년은 침묵하며
기다리고 있었지만, 프란츠 크로머는 좁은 눈으로
나를 뚫어지게 쳐다보았다. “그 이야기가 사실이야?” 그가
위협적인 어조로 물었다.
“네,” 내가 대답했다.
“정말, 진심으로?”
“네, 정말로 사실이에요,” 나는
속으로는 두려움에 숨이 막힐 지경이었지만, 도전적으로 단언했다.
“맹세할 수 있겠어?”
나는 무척 두려웠지만 망설임 없이 ‘네’라고 대답했다.
“가슴에 손을 얹고?”
“가슴에 손을 얹고,”
“좋아,” 그가 말하고는 등을 돌렸다.
나는 이 모든 상황이 아주 만족스러웠고 기뻤다.
15

In [ ]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

In [ ]:
!pip install tiktoken

## Step3 : TextEmbedding 사용해보기

In [ ]:
import openai

In [ ]:
client = openai.OpenAI()

In [ ]:
for model in client.models.list():
    if "embedding" in model.id:
        print(model.id)

text-embedding-3-large
text-embedding-ada-002
text-embedding-3-small


In [ ]:
from langchain_openai import OpenAIEmbeddings


embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

## Step4 : VectorStore 사용해보기

In [ ]:
!pip install chromadb

In [ ]:
!pip install langchain-chroma

In [ ]:
from langchain_chroma import Chroma

In [ ]:
!pip install --upgrade opentelemetry-api
!pip install --upgrade opentelemetry-sdk

제일 처음에 사용했던, PDF를 다시 사용하도록 합니다!

In [ ]:
# 위에서 사용했던 코드입니다
loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

Chroma에 임베딩 시킵니다

In [ ]:
db = Chroma.from_documents(docs, embedding_model)

이제 쿼리를 날려보겠습니다

In [ ]:
query = "how Demian look like?"
docs = db.similarity_search(query)

In [ ]:
print(docs[0].page_content)

DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a very 
uncanny sensation in my remote consciousness. I saw 
Demian's face and remarked that it was not a boy's face 
but a man's and then I saw, or rather became aware, that 
it was not really the face of a man either; it had some­
thing different about it, almost a feminine element. And 
for the time being his face seemed neither masculine 
nor childish, neither old nor young but a hundred years 
old, almost timeless and bearing the mark of other 
periods of history than our own. Animals might look 
thus, trees or stars. I did not know then, of course, I 
did not feel exactly what I am writing a

```메스꺼움을 느끼며 나는 데미안의 표정을 눈치챘다.
그는 앞으로 나서지 않고 맨
뒤에 서서, 여느 때처럼 우아하고 여유로운 모습이었다. 그의
시선은 말의 머리를 향하고 있는 듯했고, 거기에는 다시금
깊고 고요하며, 거의 광신적이면서도 열정적인
몰입이 드러나 있었다. 나는 잠시 동안 그를 빤히 쳐다보지 않을 수 없었고,
바로 그때 내 깊은 의식 속에서 매우
기이한 감각을 느꼈다. 나는
데미안의 얼굴을 보았고, 그것이 소년의 얼굴이 아니라
남자의 얼굴이라는 것을 깨달았다. 그리고 나서 나는,
아니 오히려 깨닫게 되었는데, 그것이 사실 남자의 얼굴도 아니라는 것을; 그 얼굴에는 뭔가
무언가 다른 점이 있었고, 거의 여성적인 요소가
담겨 있었다. 그리고
그 순간 그의 얼굴은 남성적이지도
아이 같지도 않았고, 늙지도 젊지도 않은, 마치 백 년
이나 된 듯한, 거의 시간을 초월한 듯하며 우리 시대와는
다른 역사적 시대의 흔적을 띠고 있는 것처럼 보였다. 동물이나
나무, 혹은 별들이 그런 모습일지도 모른다. 물론 그 당시 나는
지금, 성인이 되어 이 글을 쓰면서 느끼는 것과
정확히 같은 감정을 느끼지는 못했지만, 대략 그런 느낌이었다. 어쩌면 그는
잘생겼을지도 모르고, 어쩌면 내가 그를 매력적으로 느꼈을지도 모르고, 어쩌면
그가 나를 밀어내기도 했을지 모른다. 그조차도 확신할 수 없었다. 내가
본 것은 그가 우리 나머지 사람들과는 달랐다는 것,
그가 마치 동물이나 영혼, 혹은 환영 같았다는 것뿐이었다. 그를
설명할 수 있는 말은 오직 그가 달랐다는 것,
우리 나머지 사람들과는 상상조차 할 수 없을 만큼 달랐다는 것뿐이다. 그 외의 모든 것에 대해서는
기억이 전혀 나지 않으며, 어쩌면 내가
설명한 것조차 어느 정도는 나중에
받은 인상에서 추론한 것일지도 모른다.
내가 몇 살 더 먹은 후에야 비로소
그와 가까워질 수 있었다. 일반적인 관습과는 달리,
데미안은 다른 소년들과 함께 견진성사를 받지 않았는데,

## Step5 : Retriever 사용해보기

Retriever는 사용자의 질문을 Embedding 모델을 통해 벡터로 변환한 뒤,
VectorStore에 저장된 문서 벡터들과 비교하여
의미적으로 가장 유사한 문서(Chunk)를 찾아 반환하는 역할을 합니다.

즉, Retriever는
RAG 시스템에서 “어떤 정보를 LLM에게 참고 자료로 줄 것인가”를 결정하는 핵심 컴포넌트이며,
검색 결과의 품질이 곧 최종 답변의 품질로 이어집니다.

In [ ]:
!pip install -U langchain langchain-classic

In [ ]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

긴 문서 전체를 한 번에 LLM에 전달하는 대신,
Retriever와 LLM을 결합한 RetrievalQA 체인을 사용하여
문서에서 질문과 관련된 부분만 검색하고,
그 결과를 바탕으로 답변을 생성합니다.

이를 통해 길이가 긴 문서에서도
토큰 제한을 넘지 않으면서, 근거 기반의 질의응답을 수행할 수 있습니다.

In [ ]:
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# OpenAI 모델로 변경
# streaming=True와 callbacks 설정을 통해 실시간 출력을 활성화합니다.
llm = ChatOpenAI(
    model="gpt-6-sol",
    streaming=True,              # 실시간 출력을 켭니다
    callbacks=[StreamingStdOutCallbackHandler()], # 출력을 콘솔에 바로 뿌려줍니다
)

체인의 종류와 검색(Retrieval) 방식,
그리고 그에 따른 주요 파라미터를 설정합니다.

이 단계에서는
Retriever가 어떤 전략으로 문서를 검색할지,
그리고 몇 개의 문서를 LLM에게 전달할지를 결정하게 됩니다.
이 선택은 최종 답변의 품질과 직접적으로 연결됩니다.

예를 들어,
MMR(Maximal Marginal Relevance) 방식은
쿼리와의 유사도뿐만 아니라 문서 간의 중복을 줄이고 다양성을 확보하는 재정렬(Re-ranking) 전략입니다.

실무 환경에서는 단일 문서에 정보가 몰리는 것을 방지하고,
LLM이 보다 풍부한 문맥을 참고하도록 하기 위해
MMR 방식이 자주 사용됩니다.

In [ ]:
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

위 코드에서 짚고 넘어갈 파라미터는 다음과 같습니다  
🔹 chain_type="stuff"

검색된 문서(Chunk)를 그대로 하나의 Prompt에 모두 삽입하는 방식입니다.
구조가 단순하고 이해하기 쉬워,
RAG 구조를 처음 학습하거나 프로토타입을 만들 때 적합합니다.
단점으로는 문서 수가 많아질 경우
토큰 사용량이 빠르게 증가할 수 있습니다.
실무에서는 초기 검증 단계에서는 stuff를,
문서 수가 많아지면 map_reduce나 refine 방식으로 확장합니다.  

🔹 retriever

VectorStore에서 어떤 문서를 검색할지 결정하는 검색 모듈입니다.
검색 전략과 파라미터 설정에 따라 LLM이 참고하는 정보의 범위와 품질이 달라집니다.  

🔹 search_type="mmr"

MMR(Maximal Marginal Relevance) 검색 방식을 사용합니다. 쿼리와의 유사도뿐만 아니라, 문서 간 중복을 줄여 다양한 문맥을 확보하는 Re-ranking 전략입니다. 실무 환경에서 단일 문서 편향을 줄이기 위해 자주 사용됩니다

🔹 search_kwargs={"k": 3, "fetch_k": 10}  
- fetch_k  
VectorStore에서 우선적으로 가져올 후보 문서 개수입니다. Re-ranking 이전 단계에서 사용됩니다.
- k  
최종적으로 LLM에게 전달할 문서(Chunk)의 개수입니다.

일반적으로 fetch_k > k 로 설정하여 후보 풀을 넉넉히 확보한 뒤, 품질 좋은 문서만 선별하는 방식을 사용합니다.  

🔹 return_source_documents=True

답변 생성에 사용된 원문 문서(Chunk)를 함께 반환합니다. 이를 통해 답변의 출처를 사용자에게 표시하거나 검색 품질을 디버깅하고 RAG 성능을 평가할 수 있습니다. 실무 서비스에서는 거의 필수적으로 사용하는 옵션입니다.

In [ ]:
query = "how demian looks like? "
result = qa(query)

Sinclair sees Demian as elegant and at ease, with an unusually timeless face. It seems neither clearly young nor old, masculine nor feminine. More than any specific feature, what strikes Sinclair is how different Demian looks from everyone else.

마크다운 형식으로 출력해봅니다

In [ ]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

Sinclair sees Demian as elegant and at ease, with an unusually timeless face. It seems neither clearly young nor old, masculine nor feminine. More than any specific feature, what strikes Sinclair is how different Demian looks from everyone else.

In [ ]:
translation_request = llm.invoke(f"Translate the following English text to Korean: {result['result']}")
display(Markdown(translation_request.content))

싱클레어가 보기에 데미안은 우아하고 태연하며, 유난히 세월을 초월한 듯한 얼굴을 하고 있다. 그 얼굴은 젊은지 나이 든 건지, 남성적인지 여성적인지 뚜렷하지 않다. 싱클레어에게 무엇보다 인상적인 것은 특정한 생김새가 아니라 데미안이 다른 모든 사람과는 확연히 달라 보인다는 점이다.

싱클레어가 보기에 데미안은 우아하고 태연하며, 유난히 세월을 초월한 듯한 얼굴을 하고 있다. 그 얼굴은 젊은지 나이 든 건지, 남성적인지 여성적인지 뚜렷하지 않다. 싱클레어에게 무엇보다 인상적인 것은 특정한 생김새가 아니라 데미안이 다른 모든 사람과는 확연히 달라 보인다는 점이다.

1. 페이지 수 출력

In [ ]:
pdf_pages = loader.load()
print("PDF 페이지 수:", len(pdf_pages))

PDF 페이지 수: 182


2. Demian chunk 수 출력

In [ ]:
print("Demian chunk 수:", len(demian_chunks))

Demian chunk 수: 182


3. Demian chunk의 임베딩 차원 출력

In [ ]:
sample_vector = embedding_model.embed_documents(
    [demian_chunks[0].page_content]
)[0]

print("Demian chunk 임베딩 차원:", len(sample_vector))

Demian chunk 임베딩 차원: 1536


4. 기존 retriever의 검색 결과 출력

In [ ]:
retrieval_question = "Why did Damian tell the story about stealing the apple?"
retrieved_docs = qa.retriever.invoke(retrieval_question)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n[검색 결과 {i} | PDF {doc.metadata['page'] + 1}쪽]")
    print(doc.page_content)


[검색 결과 1 | PDF 13쪽]
TWO WORLDS 
to him so that I couldn't avoid l~king into h~ face at 
dose range. He had an evil gleam in his eyes and he 
gave an ugly laugh. His face was full of cruelty and 
sense of power. 
"Yes, kid, I can tell you who owns the orchard. I've 
known for a long time that people have been stealing 
apples and I also know that the man in question said he 
would give two marks reward to anyone who could tell 
him who stole the fruit." 
"Heavens I " I exclaimed. "But you wouldn't let on to 
him?" 
I felt that it would be futile to appeal to his sense of 
honour. He belonged to the 'other' world; betrayal was 
no aime as far as he was concerned. That much was 
dear to me. The people of the •other' world were not 
like us in these matters. 
"Not let on! My dear fellow, do you think I can mint 
my own money and produce a couple of marks out of a 
hat? I'm poor. I haven't a rich father like you and if I 
can earn two marks I've got to earn them. He might even 
give me mor

```
### 검색 결과 1 | PDF 13쪽 — 두 세계

…그와 바싹 붙어 있었기에 나는 바로 눈앞에서 그의 얼굴을 마주 볼 수밖에 없었다. 그의 눈에는 사악한 빛이 번뜩였고, 그는 흉측하게 웃었다. 얼굴에는 잔인함과 힘을 쥔 자의 우월감이 가득했다.

“그래, 꼬마야. 그 과수원 주인이 누군지 내가 알려 줄 수 있지. 사람들이 사과를 훔쳐 간다는 건 오래전부터 알고 있었어. 그리고 그 주인이 사과를 훔친 놈이 누군지 알려 주는 사람에게는 사례금으로 2마르크를 주겠다고 했다는 것도 알고 있지.”

“맙소사!” 내가 외쳤다. “하지만 그 사람한테 말하지는 않을 거지?”

그의 명예심에 호소해 봐야 소용없으리라는 생각이 들었다. 그는 ‘다른’ 세계에 속해 있었다. 그에게 배신은 죄가 아니었다. 그 점만은 내게도 분명했다. ‘다른’ 세계의 사람들은 이런 일에서 우리와 달랐다.

“말하지 말라고! 이 친구야, 내가 돈을 찍어 내거나 모자 속에서 2마르크를 뚝딱 꺼낼 수 있는 줄 알아? 난 가난해. 너처럼 부자 아버지를 둔 것도 아니고. 2마르크를 벌 수 있다면 벌어야 한다고. 어쩌면 그보다 더 줄지도 모르잖아.”

그는 느닷없이 나를 두고 가 버렸다. 우리 집 복도에서는 더 이상 평온과 안식의 냄새가 나지 않았다. 내 주위의 세계가 와르르 무너져 내리고 있었다. 그는 나를 범죄자로 고발할 터였다. 아버지에게도 그 일이 알려질 것이고, 어쩌면 경찰까지 찾아올지 몰랐다. 혼돈의 온갖 공포가 나를 위협했다. 내 앞날은 끔찍하고 위험해 보였다. 실제로 도둑질을 하지 않았다는 사실 따위는 사소한 문제에 불과했다. 나는 도둑질을 했다고 맹세했으니까. 하느님 맙소사! 눈에 눈물이 고였다. 이 상황에서 벗어나려면 무언가 대가를 치러야 한다는 생각이 들었다. 나는 필사적으로 몸을 더듬어 보았다. 사과 한 알도, 주머니칼 하나도 없었다. 아무것도 없었다.

### 검색 결과 2 | PDF 29쪽 — 카인

나는 그가 무엇을 두고 하는 말인지 전혀 알 수 없었다. 그가 나보다 우리 집을 더 잘 아는 것 같아 놀라웠다. 아치 꼭대기의 쐐기돌에는 분명 문장(紋章)이 새겨져 있었지만, 수백 년의 세월 동안 닳아 밋밋해진 데다 여러 차례 덧칠까지 되어 있었다. 내가 아는 한 그것은 우리나 우리 가문의 역사와는 아무런 관계도 없었다.

“그건 나도 잘 몰라.” 나는 수줍게 말했다. “새인지 뭔지 그런 모양인데, 아주 오래된 것 같아. 옛날에는 이 집이 수도원에 속해 있었다고 하더라.”

“그럴 수도 있겠네.” 그가 고개를 끄덕이며 말했다. “한번 자세히 살펴봐. 내 생각엔 그 새는 새매 같아.”

우리는 계속 걸었다. 나는 몹시 어색하고 거북했다. 갑자기 데미안이 무슨 우스운 생각이라도 떠오른 듯 웃었다.

“참, 너희 수업할 때 나도 거기 있었어.” 그가 불쑥 말했다. “이마에 표식을 지닌 카인 이야기였지? 너는 그 이야기 좋아해?”

아니었다. 나는 우리가 배워야 하는 것들 가운데 마음에 드는 것이 거의 없었다. 하지만 그 말을 솔직히 털어놓을 용기는 나지 않았다. 어른과 이야기하고 있는 듯한 느낌이 들었기 때문이다. 나는 그 이야기가 꽤 좋다고 대답했다.

데미안이 내 등을 툭 쳤다.

“이 친구야, 내 앞에서는 그런 척하지 않아도 돼. 어쨌든 그 이야기는 정말 특별해. 수업 시간에 배우는 다른 이야기들 대부분보다 훨씬 더 그렇지. 너희 선생님은 그 이야기에 관해 별말씀을 하지 않으셨어. 하느님이니 죄니 하는 늘 하던 말씀뿐이었지. 하지만 내 생각에는…….”

그는 말을 멈추더니 미소를 지으며 물었다.

“그런데 너, 이런 얘기에 관심이 있니?”

### 검색 결과 3 | PDF 30쪽 — 데미안

“그래. 그러니까 내 생각은 이래.” 그가 말을 이었다. “이 카인 이야기는 다른 식으로 해석할 수도 있다는 거야. 우리가 배우는 이야기들은 대부분 타당하고 진실하지만, 선생님들과는 다른 관점에서 볼 수도 있거든. 그러면 그 이야기들이 훨씬 더 이치에 맞게 돼. 예를 들어 카인이나 그의 이마에 난 표식만 해도, 선생님들이 설명하는 식으로는 납득하기 어렵잖아. 그렇지 않아? 물론 사람이 다투다가 자기 형제를 죽일 수도 있어. 그러고는 겁에 질려 고개를 숙이고 잘못을 빌 수도 있겠지. 하지만 그 비겁함 때문에 특별한 ‘표식’을 받았는데, 그 표식이 일종의 보호막이 되어 다른 모든 사람에게 두려움을 불러일으킨다는 건 정말 이상한 일이야.”

“그러네.” 그 이야기에 흥미가 생긴 나는 말했다. “하지만 그 이야기를 어떻게 다르게 해석할 수 있는데?”

그는 다시 내 어깨를 툭 쳤다.

“아주 간단해. 실제로는 그 ‘표식’이 먼저 있었고, 거기서 이 이야기 전체가 생겨난 거야. 얼굴에 남들을 두렵게 하는 무언가를 지닌 사람이 있었던 거지. 사람들은 감히 그에게 손을 대지 못했어. 그와 그의 자손들에게 위압감을 느꼈던 거야. 그의 이마에 우편 소인처럼 실제 표시가 찍혀 있지는 않았을 게 거의 확실해! 실제 삶은 그렇게 조잡하지 않거든. 오히려 거의 알아차릴 수 없는 어떤 표시였겠지. 이를테면 사람들이 보통 보아 온 눈빛보다 조금 더 지적이고 침착한 눈빛 같은 거야.

그에게는 힘이 있었고, 사람들은 모두 그를 두려워했어. 그에게는 ‘표식’이 있었던 거야. 그걸 어떻게 설명하든 상관없어. 사람들은 언제나 자기에게 편하고, 자기가 옳다고 여길 수 있는 쪽을 원하거든. 사람들은 카인의 자손들을 두려워했어. 그들에게는 표식이 있었으니까. 그래서 그 표식을 있는 그대로 해석하지 않고 오히려 정반대로 해석한 거야. 그런 표식을 지닌 사람들은 이상하다고 말했지. 실제로 그들은……”

5. 기존 답변 1개와 새 질문 2개의 답변·출처 출력

In [ ]:
qa_results = [("how demian looks like?", result)]

for question in [
    "Why did Damian tell the story about stealing the apple?",
    "How does Sinclair first meet Demian?",
]:
    qa_results.append((question, qa.invoke({"query": question})))

for n, (question, answer) in enumerate(qa_results, 1):
    print(f"\n=== 질문 {n}: {question} ===")
    print("답변:", answer["result"])

    for i, source in enumerate(answer["source_documents"], 1):
        print(f"\n[출처 {i} | PDF {source.metadata['page'] + 1}쪽]")
        print(source.page_content)

The excerpt doesn’t say Demian told that story. The narrator had claimed he stole apples, though he hadn’t; another boy then threatened to report him for a reward.Sinclair first meets Max Demian at school. Demian joins him on the walk home, and they talk about the biblical story of Cain and Abel.
=== 질문 1: how demian looks like? ===
답변: Sinclair sees Demian as elegant and at ease, with an unusually timeless face. It seems neither clearly young nor old, masculine nor feminine. More than any specific feature, what strikes Sinclair is how different Demian looks from everyone else.

[출처 1 | PDF 54쪽]
DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a


## 도입부

데미안은 싱클레어가 사과를 훔쳤다는 이야기와 크로머의 협박을 자신도 알고 있으며, 싱클레어가 그 일을 혼자 감당할 필요는 없다는 것을 알려 주기 위해 그 이야기를 했다.

싱클레어는 자신보다 나이가 많은 전학생 막스 데미안을 학교에서 처음 만난다. 얼마 뒤 데미안은 그와 함께 집으로 걸어가며 성경에 나오는 카인 이야기로 대화를 시작한다. :chatgpt-content-reference{index="0"}

---

## 질문 1. 데미안은 어떻게 생겼나요?

**답변:** 데미안은 우아하고 차분한 모습이며, 갈색 머리와 인상적인 얼굴을 지녔다. 어쩌면 잘생긴 얼굴이라고도 할 수 있다. 화자에게 그의 얼굴은 유난히 성숙하고, 거의 시간을 초월한 듯 보이며, 남성적인 면과 여성적인 면이 함께 느껴진다. 때로는 미동도 하지 않는 모습 때문에 마치 석상처럼 보인다.

### 출처 1 | PDF 54쪽 — 『데미안』

> …메스꺼움을 느끼며 나는 데미안의 표정을 살폈다. 그는 앞쪽으로 비집고 나서지 않고 맨 뒤에 서 있었는데, 언제나처럼 우아하고 여유로운 모습이었다. 그의 시선은 말의 머리를 향하는 듯했고, 그 눈빛에는 또다시 깊고 조용하며, 거의 광신적이면서도 열정적인 몰입이 드러나 있었다.
>
> 나는 한동안 그를 바라보지 않을 수 없었다. 바로 그때, 내 의식의 아득한 한구석에서 아주 기묘하고 섬뜩한 감각을 느꼈다. 나는 데미안의 얼굴을 보며 그것이 소년의 얼굴이 아니라 어른 남자의 얼굴이라는 것을 알아차렸다. 그러고는 그것이 실제로는 남자의 얼굴도 아니라는 것을 보았다. 아니, 깨달았다고 해야 할 것이다. 그 얼굴에는 다른 무언가가, 거의 여성적인 요소라고 할 만한 것이 있었다.
>
> 그 순간 그의 얼굴은 남자답지도 어린아이답지도 않았고, 늙지도 젊지도 않았다. 오히려 백 년이나 된 얼굴처럼, 거의 시간을 초월한 듯했으며, 우리가 사는 시대가 아닌 다른 역사적 시대들의 흔적을 지니고 있었다. 동물이나 나무, 별들이라면 그런 모습을 하고 있을지도 몰랐다.
>
> 물론 그때의 나는 그런 것을 알지 못했다. 어른이 된 지금 이렇게 글로 쓰는 것과 정확히 같은 방식으로 느꼈던 것은 아니지만, 대체로 그와 비슷한 무엇이었다. 어쩌면 그는 잘생겼는지도 모른다. 어쩌면 나는 그에게 끌렸는지도 모른다. 어쩌면 동시에 거부감을 느꼈는지도 모른다. 그조차 확신할 수 없었다.
>
> 내가 알 수 있었던 것은 그가 우리 나머지와 다르다는 사실뿐이었다. 그는 동물이나 영혼, 혹은 하나의 형상 같았다. 그가 우리와 다르다는 것, 상상할 수 없을 만큼 다르다는 것 말고는 달리 묘사할 수 없다. 그 밖의 일은 전혀 기억나지 않는다. 어쩌면 지금 묘사한 것조차 어느 정도는 나중에 받은 인상들에서 끌어온 것인지도 모른다.
>
> 내가 그와 가까이 지내게 된 것은 그로부터 몇 년이 더 지난 뒤였다. 보통의 관례와는 달리, 데미안은 다른 소년들과 함께 입교식을 치르지 않았는데…

*본문에 표시된 쪽수: 56쪽*

### 출처 2 | PDF 90쪽 — 『데미안』

> 왜 이제야 알아차렸을까! 그것은 데미안의 얼굴이었다.
>
> 나중에 나는 종이에 그려진 얼굴을 내 기억 속 데미안의 이목구비와 자주 비교해 보았다. 둘은 분명 닮았지만 똑같지는 않았다. 그런데도 의심할 여지 없이 그것은 데미안이었다.
>
> 초여름의 어느 날 저녁, 서쪽을 향한 내 창문으로 붉은 햇빛이 비스듬히 들어왔다. 방 안은 어둑했다. 나는 베아트리체, 혹은 데미안의 그림을 창살에 붙여 놓고, 그 뒤로 햇빛이 비칠 때 어떤 효과가 나타나는지 보고 싶어졌다.
>
> 얼굴의 윤곽은 흐릿해졌지만, 가장자리가 분홍빛으로 물든 눈과 환한 이마, 생기 넘치는 붉은 입술은 그림 표면에서 가슴을 두근거리게 하는 빛을 발했다. 그림이 희미해진 뒤에도 나는 오랫동안 그 맞은편에 앉아 있었다.
>
> 그러다 차츰 그것이 베아트리체도 데미안도 아닌, 나 자신이라는 느낌이 들었다. 그림이 나를 닮았다는 뜻은 아니었다. 나는 그림이 나를 닮아야 한다고 생각하지도 않았다. 그러나 그 얼굴은 어쩐지 내 삶을 표현하고 있었다. 그것은 내 내면의 자아였고, 나의 운명이었으며, 나의 다이몬이었다.
>
> 언젠가 내가 친구를 다시 찾게 된다면 그는 그런 모습일 터였다. 내가 사랑하는 여인도, 언젠가 그런 연인이 생긴다면, 바로 그런 모습일 터였다. 그것은 내 삶과 죽음의 형상이었으며, 내 운명의 음색과 리듬을 표현하고 있었다.
>
> 그 몇 주 동안 나는 독서에 몰두하기 시작했는데, 그 책은 그때까지 읽었던 어떤 책보다도 내게 깊은 인상을 남겼다. 나중의 삶에서도 책에 그토록 완전히 빠져든 일은 드물었다. 어쩌면 니체의 책들마저 예외가 아니었을 것이다. 그것은 노발리스의 책 한 권이었는데…
>
> **인용문에 실린 옮긴이 주:** 소설 『데미안』은 처음에 에밀 싱클레어라는 필명으로 출간되었다. 에밀 싱클레어는 헤세가 매우 존경했던 시인 노발리스의 친구 이름이다.

*본문에 표시된 쪽수: 91쪽*

### 출처 3 | PDF 71쪽 — 「도둑」

> …내 눈은 그 창백한 돌가면에 홀린 듯 못 박혀 있었다. 그리고 바로 여기에 진짜 데미안이 있다고 느꼈다!
>
> 이전에 나와 함께 다니며 이야기를 나누던 그는 겨우 그의 절반이었을 뿐이다. 잠시 어떤 역할을 맡아 그 역할에 자신을 맞추고, 나를 위해 놀이에 동참해 주던 사람에 지나지 않았다.
>
> 그러나 진짜 데미안은 바로 이런 모습이었다. [원문 일부 판독 불명] 아득히 오래된 존재 같고, 동물 같고, 돌 같고, 아름답고, 차갑고, 죽어 있으면서도 그 안에는 섬뜩할 만큼 은밀한 생명이 깃들어 있었다. 그리고 그의 주위에는 고요한 공허의 기운이 감돌았다. 에테르와 천상의 공간, 고독한 죽음의 기운이!
>
> ‘이제 그는 완전히 자기 안으로 물러나 버렸구나.’
>
> 나는 몸을 떨며 생각했다. 그토록 외롭다고 느낀 적은 없었다. 나는 그의 존재에 조금도 참여할 수 없었다. 그는 내가 닿을 수 없는 곳에 있었다. 세상에서 가장 외딴 섬에 있는 것보다도 더 멀리 떨어져 있었다.
>
> 이런 모습을 알아차리는 사람이 나뿐이라는 사실을 도무지 이해하기 어려웠다. 모두 그를 바라보고 있어야 하고, 모두 몸을 떨고 있어야 할 것 같았다! 그러나 아무도 그에게 주의를 기울이지 않았다.
>
> 그는 조각상처럼, 우상처럼 뻣뻣하게 앉아 있었다. 파리 한 마리가 그의 이마에 내려앉아 코와 입술 위를 기어 다녔는데도 얼굴 근육 하나 움직이지 않았던 것을 나는 잊을 수 없다.
>
> 그는 어디로 물러난 것일까? 무슨 생각을 하고, 무엇을 느끼고 있었을까? 천국에 있었을까, 지옥에 있었을까? 답할 수 없는 물음이었다.
>
> 수업이 끝날 무렵, 그가 다시 숨을 쉬고 살아 움직이는 모습을 보았다. 그의 시선이 나와 마주쳤을 때, 그는 이전과 다를 바 없어 보였다. 어디에서 돌아온 것일까? 어디에 있었던 것일까?
>
> 그는 피곤해 보였다. 얼굴에는 혈색이 돌아왔고 손도 다시 움직였지만, 갈색 머리카락은 어쩐지 윤기도 생기도 없어 보였다.

*본문에 표시된 쪽수: 75쪽*

---

## 질문 2. 데미안은 왜 사과를 훔쳤다는 이야기를 했나요?

**답변:** 데미안은 싱클레어가 사과를 훔쳤다는 이야기와 크로머의 협박을 자신도 알고 있으며, 싱클레어가 그 일을 혼자 감당할 필요는 없다는 것을 알려 주기 위해 그 이야기를 했다.

### 출처 1 | PDF 13쪽 — 「두 세계」

> …그와 너무 가까이 붙어 있어서 그의 얼굴을 바로 앞에서 바라볼 수밖에 없었다. 그의 눈에는 사악한 빛이 번뜩였고, 그는 흉측하게 웃었다. 그의 얼굴에는 잔인함과 힘을 쥐었다는 자신감이 가득했다.
>
> “그래, 꼬마야. 그 과수원 주인이 누군지 알려 줄 수 있어. 사람들이 그곳에서 사과를 훔쳐 간다는 건 오래전부터 알고 있었어. 게다가 그 주인이 누가 과일을 훔쳤는지 알려 주는 사람에게 보상으로 2마르크를 주겠다고 했다는 것도 알고 있지.”
>
> “세상에!” 나는 외쳤다. “하지만 그 사람에게 말하지는 않을 거지?”
>
> 그의 명예심에 호소해 봐야 소용없다는 것을 느꼈다. 그는 ‘저쪽’ 세계에 속한 사람이었다. 그에게 배신은 죄가 아니었다. 그 점만큼은 분명했다. ‘저쪽’ 세계의 사람들은 이런 문제에서 우리와 달랐다.
>
> “말하지 말라고? 이 친구야, 내가 돈을 직접 찍어내거나 모자에서 2마르크를 뚝딱 꺼낼 수 있는 줄 알아? 난 가난해. 너처럼 부자 아버지가 없다고. 그러니 2마르크를 벌 수 있다면 벌어야지. 어쩌면 그 사람이 더 줄지도 몰라.”
>
> 그는 갑자기 나를 두고 떠났다. 우리 집 현관 통로에서는 더 이상 평온함과 안전함의 냄새가 나지 않았다. 내 세상이 무너져 내리고 있었다.
>
> 그는 나를 범죄자로 고발할 것이다. 사람들은 아버지에게 알릴 것이고, 어쩌면 경찰이 올지도 모른다. 혼돈의 온갖 공포가 나를 위협했다. 내 앞날은 끔찍하고 위험해 보였다.
>
> 내가 실제로 도둑질을 하지 않았다는 사실은 사소한 일에 불과했다. 내가 그랬다고 맹세해 버렸으니까. 하늘에 계신 하나님! 눈에 눈물이 고였다.
>
> 돈을 주고 이 상황에서 벗어나야만 할 것 같았다. 나는 필사적으로 무언가를 더듬어 찾았다. 사과도, 주머니칼도, 아무것도 없었다.

*본문에 표시된 쪽수: 15쪽*

### 출처 2 | PDF 29쪽 — 「카인」

> 나는 그가 무엇을 가리키는지 전혀 몰랐다. 그가 우리 집을 나 자신보다도 더 잘 아는 것 같아 놀랐다.
>
> 아치의 머릿돌에는 분명 문장(紋章)이 새겨져 있었다. 그러나 수백 년 동안 닳고 닳아 매끈해진 데다가, 그 위에 여러 차례 덧칠까지 되어 있었다. 내가 아는 한 그것은 우리나 우리 가문의 역사와 아무런 관계도 없었다.
>
> “그건 잘 몰라.” 나는 수줍게 말했다. “새인지, 그런 종류일 거야. 아주 오래된 것 같아. 옛날에는 이 집이 수도원 소유였대.”
>
> “그럴 수도 있겠네.” 그는 고개를 끄덕이며 말했다. “한번 자세히 봐. 내 생각에는 그 새가 새매인 것 같아.”
>
> 우리는 계속 걸었다. 나는 몹시 어색하고 주눅이 들었다. 갑자기 데미안이 무슨 우스운 생각이라도 떠오른 듯 웃었다.
>
> “맞아, 네가 수업을 받을 때 나도 거기 있었어.” 그가 불쑥 말했다. “이마에 표식을 가진 카인 이야기였지? 그 이야기가 마음에 들어?”
>
> 아니었다. 수업에서 배우는 내용이 마음에 드는 경우는 드물었다. 그러나 감히 그렇게 말할 수는 없었다. 어른이 내게 말을 거는 것처럼 느껴졌기 때문이다. 나는 그 이야기가 제법 마음에 든다고 말했다.
>
> 데미안은 내 등을 툭 쳤다.
>
> “내 앞에서는 꾸밀 필요 없어, 친구. 그렇지만 그 이야기는 정말 놀라워. 수업에서 배우는 다른 이야기들 대부분보다 훨씬 더 그렇지. 너희 선생님은 그 이야기에 관해 별말씀을 하지 않으셨어. 하나님이니 죄니 하는, 늘 하시는 말씀뿐이었지. 하지만 내 생각에는…”
>
> 그는 말을 멈추고 웃으며 물었다.
>
> “그런데 너, 관심 있어?”

### 출처 3 | PDF 30쪽 — 『데미안』

> “응. 그러니까 내 생각에는,” 그가 말을 이었다. “이 카인 이야기는 다르게 해석할 수 있어. 우리가 배우는 이야기 대부분은 나름대로 타당하고 진실해. 하지만 선생님들과는 다른 관점에서 바라볼 수도 있고, 그러면 그 의미가 훨씬 더 분명해지지.
>
> 예를 들어 카인과 그의 이마에 있는 표식에 관해 선생님들이 설명하는 내용만으로는 만족할 수 없잖아. 그렇지 않아?
>
> 누군가 다투다가 자기 형제를 죽일 수도 있어. 그러고 나서 겁을 먹고 비굴하게 용서를 빌 수도 있지. 하지만 그 비겁함 때문에 특별한 ‘표식’을 받는데, 그 표식이 일종의 보호막이 되어 주고 다른 사람들을 두렵게 한다는 건 정말 이상하잖아.”
>
> “그렇네.” 나는 그 주제에 관심이 생겨 말했다. “하지만 그 이야기를 달리 어떻게 해석할 수 있는데?”
>
> 그는 다시 내 어깨를 툭 쳤다.
>
> “아주 간단해. 실제로 일어났던 일, 그리고 그 이야기 전체가 생겨난 근원에는 바로 그 ‘표식’이 있었던 거야.
>
> 다른 사람들을 두렵게 만드는 무언가를 얼굴에 지닌 사람이 있었던 거지. 사람들은 감히 그에게 손을 대지 못했어. 그와 그의 자식들은 사람들에게 강렬한 인상을 주었지.
>
> 그의 이마에 우편물에 찍힌 도장 같은 실제 표식이 있었던 것은 거의 확실히 아니야! 현실의 삶은 그렇게 조잡하지 않거든.
>
> 오히려 거의 알아차리기 어려운 어떤 특징이 있었던 거야. 사람들이 익숙하게 보아 온 것보다 조금 더 많은 지성과 침착함이 그의 눈에 담겨 있었던 거지.
>
> 그 사람에게는 힘이 있었고, 사람들은 모두 그를 두려워했어. 그에게는 ‘표식’이 있었던 거야. 그걸 어떻게 설명하든 상관없어.
>
> 사람들은 언제나 자기 마음이 편해지고, 자신이 옳다고 여길 수 있는 설명을 바라지. 사람들은 카인의 자손들을 두려워했어. 그들에게는 표식이 있었으니까.
>
> 그래서 그 표식은 실제 의미대로 해석되지 않고, 정반대로 해석되었어. 사람들은 그런 표식을 가진 이들이 이상하다고 말했지. 사실 그들은…”

*본문에 표시된 쪽수: 31쪽*

---

## 질문 3. 싱클레어는 데미안을 어떻게 처음 만나나요?

**답변:** 싱클레어는 자신보다 나이가 많은 전학생 막스 데미안을 학교에서 처음 만난다. 얼마 뒤 데미안은 그와 함께 집으로 걸어가며 성경에 나오는 카인 이야기로 대화를 시작한다.

### 출처 1 | PDF 146쪽 — 『데미안』

> “일본에서도 사정이 더 나을 거라고는 생각하지 않네. 곧장 난롯가로 달려가지 않는 사람들은 온 세상을 통틀어 드물지. 이곳에도 그런 사람들이 좀 있기는 해.”
>
> 그가 하는 말이 하나하나 내 마음을 파고들 때마다 기쁨과 불안이 뒤섞인 감정이 일었다. 나는 그 말을 하는 사람을 알고 있었다. 데미안이었다.
>
> 나는 거센 바람이 부는 밤, 어두운 골목길을 따라 그와 일본인의 뒤를 걸으며 그들의 대화에 귀를 기울였다. 데미안의 목소리는 내 귀에 음악처럼 들렸다. 익숙한 울림, 익숙한 자신감과 침착함, 그리고 예전에 내게 미쳤던 그 모든 힘이 그대로 담겨 있었다.
>
> 모든 것이 다시 제자리를 찾았다. 나는 그를 찾아낸 것이다.
>
> 그들이 변두리의 어느 거리 끝에 이르렀을 때, 일본인은 작별 인사를 하고 한 집의 문을 열었다. 데미안은 왔던 길을 되돌아갔다. 나는 길 한가운데 멈춰 서서 그를 기다리고 있었다.
>
> 그가 탄력 있는 걸음걸이와 곧게 편 자세로 내게 다가오는 모습을 보자 심장이 빠르게 뛰었다. 그는 황갈색 방수 외투를 입고 있었고, 가느다란 지팡이 하나가 팔에 매달려 흔들리고 있었다.
>
> 그는 거의 내 앞에 이를 때까지 한결같은 속도로 걸어왔다. 그러고는 모자를 벗었다. 나는 단호한 입매와 넓은 이마의 독특한 광채를 지닌, 익숙하고 지적인 얼굴을 더 또렷하게 볼 수 있었다.
>
> “데미안!” 내가 소리쳤다.
>
> 그는 내게 손을 내밀었다.
>
> “너였구나, 싱클레어! 널 기다리고 있었어.”
>
> “내가 여기 있다는 걸 알고 있었어?”
>
> “정확히 알았던 건 아니지만, 네가 여기 있기를 바랐지. 널 본 건 오늘 밤이 처음이야. 너는 줄곧 우리 뒤를 따라오고 있었잖아.”
>
> “그럼, 날 바로 알아본 거야?”

*본문에 표시된 쪽수: 148쪽*

### 출처 2 | PDF 182쪽 — 『데미안』

> “…그리고 또 한 가지가 있어. 에바 부인이, 네가 혹시 곤경에 처하게 되면 내게 주신 입맞춤을 너에게 전해 달라고 하셨어.… 눈을 감아, 싱클레어.”
>
> 나는 그의 말대로 눈을 감았다. 좀처럼 줄어들지 않는 핏방울이 맺힌 내 입술에, 입맞춤이 가볍게 스치는 것을 느꼈다. 그러고는 잠이 들었다.
>
> 다음 날 아침 사람들이 나를 깨웠다. 상처를 치료하고 붕대를 갈아야 했다. 마침내 정신이 제대로 들자, 나는 재빨리 내 옆의 매트리스 쪽으로 고개를 돌렸다. 거기에는 한 번도 본 적 없는 낯선 사람이 누워 있었다.
>
> 상처를 치료하는 일은 고통스러웠다. 그 뒤에 내게 일어난 모든 일도 마찬가지였다.
>
> 그러나 그처럼 수많은 순간에 내가 열쇠를 찾아, 어두운 거울 속에서 운명의 형상들이 잠들어 있는 내면 깊은 곳을 들여다볼 때면, 그 검은 거울 위로 고개를 숙이기만 하면 된다.
>
> 그러면 이제는 나의 친구이자 인도자인 그와 완전히 닮아 있는, 나 자신의 모습을 볼 수 있다.

### 출처 3 | PDF 130쪽 — 『데미안』

> …그것이 무엇인지.
>
> 나는 창문에 드문드문 불이 켜진 매음굴들이 있는 변두리를 지나갔다. 더 외곽으로 나가자 새 건물 몇 채와 잿빛 눈에 일부가 덮인 벽돌더미들이 있었다.
>
> 이 이상한 충동에 이끌려 몽유병자처럼 떠돌던 나는 고향의 새 건물을 알아보았다. 그곳은 나를 괴롭히던 크로머가 처음 돈 문제를 정산하기 위해 나를 데려갔던 곳이었다.
>
> 그와 비슷한 건물이 잿빛 밤 속에서 내 앞에 서 있었다. 크게 벌어진 출입구로 나를 향해 하품을 하는 것 같았다. 그 건물은 나를 안으로 끌어당겼다. 내게는 선택의 여지가 없었다. 나는 모래더미와 쓰레기에 걸려 비틀거리며 앞으로 나아갔다.
>
> 그 충동은 거역할 수 없었다. 나는 들어가야만 했다. 널빤지와 깨진 벽돌을 넘어 비틀거리며 황량한 방 안으로 들어갔다. 축축한 냉기와 돌 냄새가 우울하게 감돌았다. 모래더미 하나가 희미한 잿빛 얼룩처럼 보였고, 나머지는 모두 어두웠다.
>
> 그때 겁에 질린 목소리가 외쳤다.
>
> “세상에, 싱클레어! 너 어디서 온 거야?”
>
> 어둠 속에서 한 사람이 일어났다. 유령처럼 작고 야윈 사내였다. 머리카락이 곤두섰다. 나는 학교 친구 크나우어라는 것을 알아보았다.
>
> “어떻게 여기 온 거야?” 그는 격한 감정에 거의 제정신이 아닌 듯 물었다. “어떻게 날 찾아낸 거야?”
>
> 나는 그가 무슨 뜻으로 하는 말인지 이해하지 못했다.
>
> “널 찾고 있었던 건 아니야.”
>
> 나는 멍한 상태로 말했다. 한마디를 할 때마다 힘을 들여야 했다. 말들은 마치 얼어붙기라도 한 듯 무거운 입술 사이를 힘겹게 비집고 나왔다.
>
> 그는 나를 빤히 바라보았다.
>
> “날 찾은 게 아니라고?”
>
> “아니. 난 이곳에 끌려 왔어. 네가 날 불렀니? 틀림없이 네가 날 불렀을 거야. 그런데 여기서 뭘 하고 있는 거야? 지금 밤이잖아.”

*인용문들에 표시된 다운로드 출처: Holybooks.*

6. 검색 설정 비교

In [ ]:
comparison_question = "How is Demian's face described?"
similarity_retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

comparison_results = {}
for name, retriever_to_check in [
    ("기존 MMR, k=3", qa.retriever),
    ("비교 similarity, k=3", similarity_retriever),
]:
    print(f"\n=== {name} ===")
    docs = retriever_to_check.invoke(comparison_question)
    comparison_results[name] = docs
    for i, doc in enumerate(docs, 1):
        print(f"\n[결과 {i} | PDF {doc.metadata['page'] + 1}쪽]")
        print(doc.page_content)


=== 기존 MMR, k=3 ===

[결과 1 | PDF 54쪽]
DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a very 
uncanny sensation in my remote consciousness. I saw 
Demian's face and remarked that it was not a boy's face 
but a man's and then I saw, or rather became aware, that 
it was not really the face of a man either; it had some­
thing different about it, almost a feminine element. And 
for the time being his face seemed neither masculine 
nor childish, neither old nor young but a hundred years 
old, almost timeless and bearing the mark of other 
periods of history than our own. Animals might look 
thus, trees or stars. I did not know then, of course, I 
d

```
기존 MMR, k=3
결과 1 | PDF 54쪽
속이 메스꺼운 가운데, 나는 데미안의 표정을 눈여겨보았다. 그는 앞으로 나서지 않고 맨 뒤에 서 있었는데, 평소처럼 세련되고 여유로운 모습이었다. 그의 시선은 말의 머리를 향하고 있는 듯했고, 거기에는 다시금 깊고 고요한, 거의 광신적이면서도 열정적인 몰입이 드러나 있었다.
나는 잠시 그에게서 눈을 뗄 수 없었다. 바로 그때, 의식의 아득한 곳에서 아주 기묘한 감각이 일어나는 것을 느꼈다. 데미안의 얼굴을 바라보며, 그것이 소년의 얼굴이 아니라 어른 남자의 얼굴이라는 생각이 들었다. 그러다가 그것이 사실은 남자의 얼굴도 아니라는 것을 보았다. 아니, 깨달았다고 해야 할 것이다. 그 얼굴에는 무언가 다른 것이, 거의 여성적인 요소라고 할 만한 것이 있었다.
그 순간 그의 얼굴은 남성적이지도, 어린아이 같지도 않았다. 늙지도 젊지도 않았지만, 백 년을 살아온 듯했고, 거의 시간을 초월한 듯했으며, 우리 시대와는 다른 역사적 시대들의 흔적을 지니고 있었다. 동물이나 나무, 별들이라면 그런 모습을 하고 있을지도 몰랐다.
물론 그때 내가 이런 사실을 분명하게 알고 있었던 것은 아니다. 어른이 된 지금 이렇게 적고 있는 그대로를 당시에도 정확히 느꼈던 것은 아니지만, 어쨌든 그와 비슷한 느낌이었다. 어쩌면 그는 잘생겼을 수도 있고, 내가 그에게 매력을 느꼈을 수도 있으며, 동시에 거부감을 느꼈을 수도 있다. 그조차 확실히 알 수 없었다.
내가 알 수 있었던 것은 그가 우리 나머지와는 다르다는 사실뿐이었다. 그는 동물이나 영혼, 혹은 하나의 형상과도 같았다. 그가 우리와는 달랐다는 것, 상상할 수 없을 만큼 달랐다는 것 외에는 달리 표현할 수 없다. 그 밖의 모든 일에 대한 기억은 비어 있다. 어쩌면 지금 묘사한 것조차 어느 정도는 나중에 받은 인상들을 모아 구성한 것인지도 모른다.
내가 그와 가까워진 것은 그로부터 몇 년이 더 흐른 뒤였다. 일반적인 관례와 달리, 데미안은 다른 소년들과 함께 견진례를 받지 않았다….
결과 2 | PDF 90쪽
나는 왜 이제야 그것을 알아차렸을까! 그것은 데미안의 얼굴이었다.
나중에 나는 종이에 그려진 얼굴을 기억 속 데미안의 생김새와 자주 비교해 보았다. 둘은 분명 닮아 있었지만, 똑같지는 않았다. 그러나 의심할 여지 없이 그것은 데미안이었다.
초여름 어느 날 저녁, 서쪽으로 난 창문을 통해 붉은 햇살이 비스듬히 비쳐 들었다. 방 안은 어스름했다. 문득 베아트리체, 혹은 데미안의 그림을 창살에 붙여 놓고 햇빛이 비치면 어떻게 보이는지 살펴보고 싶어졌다.
얼굴의 윤곽은 흐릿해졌지만, 분홍빛으로 테두리가 물든 눈과 환한 이마, 생기 넘치는 붉은 입술은 그림 표면에서 마음을 설레게 할 만큼 빛났다. 그림이 희미해진 뒤에도 나는 오랫동안 그 맞은편에 앉아 있었다.
그러자 차츰 그것이 베아트리체도 데미안도 아닌, 바로 나 자신이라는 느낌이 밀려왔다. 그림이 내 외모를 닮았다는 말은 아니다. 닮아야 한다고 생각하지도 않았다. 다만 그 얼굴은 어떤 식으로든 내 삶을 표현하고 있었다. 그것은 내 내면의 자아였고, 내 운명, 혹은 나의 다이몬(daimon)이었다.
언젠가 내 친구를 다시 찾게 된다면, 그는 이런 모습일 터였다. 내게 사랑하는 여인이 생긴다면, 그녀 역시 이런 모습일 터였다. 그것은 내 삶과 죽음의 원형이었고, 내 운명의 음조와 리듬을 드러내고 있었다.
그 몇 주 동안 나는 책 한 권을 읽기 시작했는데, 그것은 그전에 읽었던 어떤 책보다도 깊은 인상을 남겼다. 훗날에도 나는 그토록 완전히 책에 빠져든 적이 거의 없었다. 어쩌면 니체의 책들조차 예외가 아니었을 것이다. 그것은 노발리스의 책 한 권으로….
원문 역자 주: 소설 『데미안』은 처음에 ‘에밀 싱클레어’라는 필명으로 출간되었다. 이 이름은 헤세가 매우 존경했던 시인 노발리스의 친구 이름이다.

결과 3 | PDF 88쪽
…
가끔 거리에서 마주치던 그 소녀의 생김새를 그려 내려고 애쓸수록, 오히려 잘되지 않았다. 마침내 나는 그 시도를 포기하고, 붓에 물감을 묻힐 때마다 저절로 떠오르는 상상과 생각을 따라 얼굴을 그리는 것으로 만족했다.
그렇게 나타난 것은 꿈속의 얼굴이었고, 나는 그것에 만족하지 못했다. 그래도 실험을 계속했다. 새로운 스케치를 그릴 때마다 내가 찾는 형상에 조금씩 가까워졌다. 비록 그것은 여전히 현실과는 거리가 멀었지만 말이다.
나는 점점 더 연필로 무심히 선을 긋고 그 안에 색을 칠하는 일에 익숙해졌다. 특별히 어떤 모델을 마음에 두지는 않았다. 내 무의식에서 종이 위로 흘러나와, 장난스럽기도 하고 진지하기도 한 낙서 속에서 형체를 얻는 것들만이 모델이 되었다.
그러던 어느 날, 거의 나 자신도 모르는 사이에 이전의 그림들보다 더 뚜렷한 무엇을 표현하는 초상화 한 점을 그렸다.
그것은 그 소녀의 얼굴이 아니었다. 이미 오래전부터 그 얼굴은 소녀의 모습을 벗어나 전혀 다른 것이 되어 있었다. 비현실적이었지만, 내게는 그 소녀 못지않게 큰 의미가 있었다.
그것은 소녀보다는 소년의 얼굴에 가까웠다. 머리카락은 그 아름다운 소녀의 옅은 금발이 아니라 붉은 기가 감도는 밤색이었다. 턱은 힘 있고 단호했으며, 입술은 장밋빛이었다. 전체적인 모습은 다소 딱딱하고 가면 같았지만, 강렬한 인상을 주었고 은밀한 내면의 삶이 있음을 암시했다.
완성된 그림 앞에 앉아 있자, 나는 기묘한 인상을 받았다. 그것은 일종의 신의 형상이나 성스러운 가면을 닮아 있었다. 반은 남성이고 반은 여성이었으며, 나이를 헤아릴 수 없었다. 확고한 의지를 품고 있으면서도 꿈에 잠겨 있었고, 얼어붙은 듯하면서도 신비롭게 살아 있었다.
이 얼굴에는 내게 전하려는 어떤 메시지가 있었다. 그것은 내게 속해 있었으며, 나에게 무언가를 요구하고 있었다. 누군가를 닮았지만, 누구인지 알 수 없었다.

In [ ]:
stored = db.get(include=["documents"])
print("입력 chunk:", len(demian_chunks))
print("Chroma 항목:", len(stored["ids"]))
print("서로 다른 본문:", len(set(stored["documents"])))

입력 chunk: 182
Chroma 항목: 910
서로 다른 본문: 182


chunk_size=500, chunk_overlap=0에서 182쪽이 182개 chunk가 됐다. 이 PDF에서는 사실상 한 페이지가 한 chunk로 유지되어, overlap의 효과를 비교한 결과는 없다. k=3은 답변에 문단 세 개를 제공하지만, 실제로 관련성이 낮은 문단도 포함됐다.  
데미안의 외모 답변은 PDF 54·71쪽이 직접 뒷받침하며, 88쪽은 초상화 묘사라 간접적이다. 사과 이야기의 직접 근거는 PDF 11쪽이고, 43쪽은 질문의 사건과 거리가 있다. ‘두 세계’는 PDF 5·7쪽이 뒷받침하지만, 175쪽의 전쟁 장면은 관련성이 낮다.  
같은 외모 질문에서 MMR은 서로 다른 3쪽을, similarity는 54쪽이 중복된 2쪽을 반환했다. 현재 출력에서는 MMR의 결과 다양성이 높다. 다만 중복 저장 여부를 확인하기 전에는 답변 품질까지 MMR이 더 좋다고 결론 내릴 수 없다.